In [ ]:
# 6. 下载训练好的模型
from google.colab import files
import os

# 打包 best.pt
!zip -j /content/model.zip /content/drive/MyDrive/yolo_runs/cghd_56cls_v2/weights/best.pt
files.download('/content/model.zip')
print('下载 best.pt 到本地, 放到 runs/detect/cghd_56cls_v2/weights/ 目录下即可使用')

In [ ]:
# 5. 续训 (从已有 best.pt 继续, 更激进的数据增强)
from ultralytics import YOLO

# 把本地的 best.pt 上传到 Google Drive 根目录，或在下面填上次训练的路径
MODEL_PATH = '/content/drive/MyDrive/yolo_runs/cghd_56cls/weights/best.pt'

model = YOLO(MODEL_PATH)
model.train(
    data='/content/cghd_yolo/cghd_dataset.yaml',
    epochs=80,           # 续训总 epoch (从上次断点继续, 如已训30则追加50)
    imgsz=640,
    batch=16,
    resume=False,        # False=加载权重重新开始; True=恢复完全相同的状态(optimizer/scheduler)
    name='cghd_56cls_v2',
    project='/content/drive/MyDrive/yolo_runs',
    lr0=0.0005,          # 续训用更低学习率
    lrf=0.01,            # 最终LR系数
    patience=15,
    warmup_epochs=3,
    cos_lr=True,
    # 强数据增强 (减少过拟合, 提升泛化)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,        # 旋转 ±10°
    translate=0.1,       # 平移
    scale=0.5,           # 缩放
    shear=2.0,           # 剪切
    perspective=0.0005,  # 透视
    flipud=0.5,          # 上下翻转
    fliplr=0.5,          # 左右翻转
    mosaic=1.0,          # 马赛克增强
    mixup=0.1,           # 混合增强
    copy_paste=0.1,      # 复制粘贴增强
    erasing=0.4,         # 随机擦除
)
print('续训完成!')

# CGHD YOLO 56类电路元件检测训练
**数据集**: GTDB-HD 手绘电路图, 2411训练+426验证, 56类
**运行前**: 把 `cghd_yolo.zip` 上传到 Google Drive 根目录

In [ ]:
# 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 解压数据集
!unzip -q /content/drive/MyDrive/cghd_yolo.zip -d /content/
!ls /content/cghd_yolo/

In [ ]:
# 3. 安装 ultralytics
!pip install -q ultralytics

In [ ]:
# 4. 初次训练 (从 yolov8n.pt 开始)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(
    data='/content/cghd_yolo/cghd_dataset.yaml',
    epochs=30, imgsz=640, batch=16,
    name='cghd_56cls', project='/content/drive/MyDrive/yolo_runs',
    patience=8, lr0=0.001, warmup_epochs=2
)
print('初训完成! 模型保存在 Google Drive > yolo_runs/cghd_56cls/')